# Revisión: addon real sin sistema molecular

Smoke manual para #240/#245/#257. **Guarda tus salidas, recarga primero la página del notebook y después reinicia el kernel** y usa el kernel de `molsyssuite@uibcdf_3.14` (aquí registrado como `Python 3 (ipykernel)`). La primera celda comprueba el intérprete exacto.
Es una integración de fuentes: Viewer en desarrollo y una copia aislada de PharmacophoreMT `f0d96ae0`. No cualifica paquetes instalados ni publica versiones.

Ejecuta las celdas en orden y conserva las salidas. Los clics se hacen en el widget vivo de Jupyter, con tu navegador remoto.

In [ ]:
from pathlib import Path
import sys
import subprocess

viewer_source = Path("/home/diego/repos@uibcdf/molsysviewer")
provider_source = Path("/tmp/msv-issue240-review-20261013/pharmacophoremt-review")
expected_python = Path("/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python")
assert Path(sys.executable) == expected_python, sys.executable
assert provider_source.is_dir(), "La copia de revisión de PharmacophoreMT debe estar disponible."
assert not any(name in sys.modules for name in ("molsysviewer", "pharmacophoremt", "molsysviewer_pharmacophoremt")), "Reinicia el kernel antes de esta celda."
sys.path.insert(0, str(provider_source))
sys.path.insert(0, str(viewer_source))

import molsysviewer as msv
import pharmacophoremt as phmt
import molsysviewer_pharmacophoremt as provider
from pharmacophoremt.interaction_site import HBAcceptorSphere, HydrophobicSphere
from molsysviewer_pharmacophoremt.runtime import ensure_runtime

assert Path(msv.__file__).resolve().is_relative_to(viewer_source)
assert Path(phmt.__file__).resolve().is_relative_to(provider_source)
assert Path(provider.__file__).resolve().is_relative_to(provider_source)
print("Python:", sys.executable)
print("Viewer commit local:", subprocess.check_output(["git", "-C", str(viewer_source), "rev-parse", "HEAD"], text=True).strip())
print("Viewer versión del módulo (metadatos de desarrollo):", msv.__version__, msv.__file__)
print("PharmacophoreMT fuente:", phmt.__version__, "f0d96ae0dd6d22152bc4422fd3e0ec125353e168")

## 1. Canvas vacío
Ejecuta la celda. Debe aparecer la bienvenida. Abre Studio/Addons: la bienvenida debe desaparecer y el panel ser accesible. Ciérralo: debe reaparecer.

In [ ]:
msv.addons.register_module("molsysviewer_pharmacophoremt")
view = msv.MolSysView(viewer_mode="standard", controls_mode="compact")
assert view._molsys is None
view

## 2. Preparar el addon sin molécula
Ejecuta la celda siguiente. Luego abre Studio/Addons → PharmacophoreMT. Deben verse dos sitios, y Render debe estar habilitado. Si el panel estaba abierto antes de preparar los datos, sal y vuelve al workspace para actualizarlo.

In [ ]:
pharmacophore = phmt.Pharmacophore(name="Revisión sin molécula")
pharmacophore.interaction_sites.extend([
    HBAcceptorSphere(center="[0.1,0.2,0.3] nm", radius="0.5 nm"),
    HydrophobicSphere(center="[1,1.1,1.2] nm", radius="0.4 nm"),
])
pharmacophore.n_interaction_sites = 2
view.shapes.add_sphere(center="[-1,0,0] nm", radius="0.3 nm", tag="caller-sphere")
view.addons.enable("pharmacophoremt")
runtime = ensure_runtime(view)
runtime.pharmacophore = pharmacophore
print("Preparado: dos sitios reales y una esfera independiente; ningún MolSys.")

Pulsa **Render**. Cierra temporalmente Studio/Addons para ver el canvas completo: deben verse los dos sitios del farmacóforo y la esfera independiente. Ejecuta la comprobación. Vuelve a abrir Studio/Addons para pulsar Clear.

In [ ]:
assert view._molsys is None
assert view.shapes.contains("caller-sphere")
assert len(runtime.rendered_tags) == 2
assert all(view.shapes.contains(tag) for tag in runtime.rendered_tags)
print("Render: dos sitios y esfera independiente, sin MolSys.")

Pulsa **Clear**. Sólo deben desaparecer los sitios del addon; la esfera independiente permanece. Ejecuta la comprobación.

In [ ]:
assert not runtime.rendered_tags
assert view.shapes.contains("caller-sphere")
print("Clear conserva la geometría del usuario.")

## 3. Ventanas independientes, escena compartida

1. Abre el popup del canvas. Abre allí PharmacophoreMT y pulsa Render: los sitios deben aparecer en ambos canvases.
2. En el canvas original navega al workspace Core/Studio. Desde el popup pulsa Clear y Render de nuevo: ambos deben funcionar.
3. Desde el original abre el popup de Studio/Addons, navega a PharmacophoreMT y pulsa Clear. Los sitios desaparecen en ambos canvases; la esfera independiente permanece.
4. Cierra **sólo el popup de Studio**. Desde el canvas popup pulsa Render otra vez: debe funcionar.
5. En el popup de canvas sal del workspace y vuelve. Render/Clear deben seguir funcionando y los dos sitios deben aparecer en la lista.
6. Cierra el canvas popup. En el original vuelve a PharmacophoreMT; Render/Clear deben seguir funcionando.

Conserva el widget abierto hasta finalizar las comprobaciones. Confirma si los seis pasos funcionan o indica el paso y lo observado.

## 4. Añadir una molécula (opcional)
Ejecuta la celda. El addon debe seguir accesible y no debe reaparecer la bienvenida encima de la escena. Render/Clear deben conservar la proteína y la esfera independiente.

In [ ]:
view.load(msv.demo["dialanine"].molsys)
assert view.shapes.contains("caller-sphere")

## Finalizar
Ejecuta sólo al terminar; libera los widgets y cualquier popup conectado.

In [ ]:
view.close()